In [ ]:
from google.colab import files
files.upload()

Saving week10_students.xlsx to week10_students (1).xlsx


{'week10_students (1).xlsx': b'PK\x03\x04\n\x00\x00\x00\x00\x00\x87N\xe2@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\t\x00\x00\x00docProps/PK\x03\x04\x14\x00\x00\x00\x08\x00\x87N\xe2@.\xce\x98&8\x01\x00\x00C\x02\x00\x00\x10\x00\x00\x00docProps/app.xml\x9d\x91\xc1J\xc40\x10\x86\xef\x82\xef\x10r\xdfM\xbb\x88\xc8\xd2f\x11D\xbcYp\xd5\xa3\xc4t\xba\x1bl\x93\x92\x99\xad\xbb>\x8b\x17\x0f\x82o\xe0\xc9\xb7Q\xf016mA\xbb\xea\xc9\xdb\xcc\xfc\xc3?\xdf\x9f$\xb3uU\xb2\x06<\x1agS\x1e\x8f#\xce\xc0j\x97\x1b\xbbH\xf9\xe5\xfctt\xc4\x19\x92\xb2\xb9*\x9d\x85\x94o\x00\xf9L\xee\xef%\x99w5x2\x80,XXL\xf9\x92\xa8\x9e\n\x81z\t\x95\xc2q\x90mP\n\xe7+E\xa1\xf5\x0b\xe1\x8a\xc2h8qzU\x81%1\x89\xa2C\x01k\x02\x9bC>\xaa\xbf\x0cy\xef8m\xe8\xbf\xa6\xb9\xd3-\x1f^\xcd7u\x00\x96\xc9q]\x97F+\n)\xe5uv\xc12\x0f\x18\x10\xbaA"\x86jr\x06\xaaM\x9f)\xe3Q&\rM\x1b\xd0\xe4<C\xf3\x10\xf2O8\xbbU\x08\xado\xca\x1b\xe5\x8d\xb2\x14\xfc\xdb\xb5\xbe\xe9\xea\xb2F\xf2\xf2\xe3\xf5\xf9\xfd\xed\xf1\xf3\xe9%\x11A\xefg]9\\\x1d\xd6\xe6@\xc6\xddB(v\x

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

# The lab uses a CSV; the same data is in this Excel file.
# To use the CSV instead: df = pd.read_csv("week10_students.csv")
df = pd.read_excel("week10_students.xlsx")

In [ ]:
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
print(df[features].corrwith(df["score"]).round(3))
# Comment: write which feature looks most useful, in your own words

X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(len(X_train), len(X_test))   # 112 38
# Comment: explain in your own words why 25% of students are held back

quiz_avg           0.801
assignments_avg    0.536
attendance_pct     0.238
dtype: float64
112 38


In [ ]:
model = LinearRegression().fit(X_train, y_train)
for name, c in zip(features, model.coef_):
    print(name, round(c, 3))
print("intercept", round(model.intercept_, 3))
# Comment: one extra quiz point is worth about 0.58 exam points

y_pred = model.predict(X_test)
print("MAE", round(mean_absolute_error(y_test, y_pred), 2))   # ~3.33
print("R2", round(r2_score(y_test, y_pred), 3))                # ~0.887
# Comment: explain MAE and R2 in plain language

new = pd.DataFrame([[70, 75, 90]], columns=features)
print("new student:", model.predict(new))                      # ~73.7
mine = pd.DataFrame([[80, 85, 95]], columns=features)          # replace with your own numbers
print("my prediction:", model.predict(mine))

quiz_avg 0.579
assignments_avg 0.365
attendance_pct 0.113
intercept -4.428
MAE 3.33
R2 0.887
new student: [73.66410989]
my prediction: [83.66863472]


In [ ]:
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xc_train, yc_train)
pred = clf.predict(Xc_test)

print("accuracy", round(accuracy_score(yc_test, pred), 3))   # 0.895
print("baseline (all pass)", round(yc_test.mean(), 3))        # 0.895
print(confusion_matrix(yc_test, pred))
# Comment: explain why accuracy alone is misleading, using the matrix

passed
1    131
0     19
Name: count, dtype: int64
accuracy 0.895
baseline (all pass) 0.895
[[ 3  1]
 [ 3 31]]


In [ ]:
def test_r2(frame):
    a, b, c, d = train_test_split(frame, y, test_size=0.25, random_state=42)
    return r2_score(d, LinearRegression().fit(a, c).predict(b))

print("quiz only:", round(test_r2(df[["quiz_avg"]]), 3))           # ~0.62
print("three features:", round(test_r2(df[features]), 3))          # ~0.89
X_prog = pd.get_dummies(df[features + ["program"]], columns=["program"], dtype=int)
print("three + program:", round(test_r2(X_prog), 3))               # ~0.88
# Comment: what helped, what didn't, and what that teaches you

quiz only: 0.624
three features: 0.887
three + program: 0.88


In [ ]:
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig2 = px.scatter(df, x="quiz_avg", y="score", color="passed",
                  hover_name="name", title="2. Who is at risk")
league = df.groupby("district")["score"].mean().reset_index()
fig3 = px.bar(league, x="district", y="score", title="3. Where to act")

fig1.show()
fig2.show()
fig3.show()

fig2.write_html("class_dashboard.html")
files.download("class_dashboard.html")   # downloads it to your computer

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
fig_b = px.box(df, x="program", y="score", color="program",
               title="Score distribution by program")
fig_b.show()
# Write two sentences in a markdown cell:
# Networking is the most consistent; Software Engineering has the widest spread.